# ThreatVision — Download & Save Model Locally
Run this notebook **once**. It will download the model from Roboflow and save it to a local directory on your machine. After this, the model never needs to be re-downloaded.

## Step 1 — Install / verify dependencies

In [ ]:
# Only run this cell if you haven't installed these yet
# !pip install inference roboflow supervision opencv-python

In [ ]:
ROBOFLOW_API_KEY = 

## Step 2 — Configuration
Set your API key and the local path where the model will be saved.

In [2]:
import os

# ── Your Roboflow API key ──────────────────────────────────────────────────────
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")   # <-- paste your key here

# ── Model info (must match config.py) ─────────────────────────────────────────
MODEL_ID      = "weapondetection-xx3lz-tgh2t/2"
WORKSPACE     = "weapondetection-xx3lz-tgh2t"
VERSION_NUM   = 2

# ── Where to save the model on your machine ───────────────────────────────────
BASE_DIR   = "/media/yousifcreates/339b14bb-e7ae-4348-b1dd-b0b8a896b600/Portfolio/Deep Learning/ThreatVision/ThreatVision"
MODEL_DIR  = os.path.join(BASE_DIR, "model")
MODEL_SAVE_PATH = os.path.join(MODEL_DIR, "weapondetection_v2")   # final folder

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print(f"Model will be saved to: {MODEL_SAVE_PATH}")

Model will be saved to: /media/yousifcreates/339b14bb-e7ae-4348-b1dd-b0b8a896b600/Portfolio/Deep Learning/ThreatVision/ThreatVision/model/weapondetection_v2


## Step 3 — Set the Roboflow cache dir & load the model

The `inference` library respects the `ROBOFLOW_MODEL_CACHE_DIR` environment variable.
Setting it **before** calling `get_model()` forces the download directly into your chosen folder.

In [3]:
# Point the inference cache at your local model directory
os.environ["ROBOFLOW_API_KEY"]         = ROBOFLOW_API_KEY
os.environ["ROBOFLOW_MODEL_CACHE_DIR"] = MODEL_SAVE_PATH

from inference import get_model

print("Downloading model from Roboflow (this only happens once)...")
model = get_model(model_id=MODEL_ID)
print("Model loaded successfully!")

ModelDependencyMissing: Your `inference` configuration does not support SAM model. Use pip install 'inference[sam]' to install missing requirements.To suppress this warning, set CORE_MODEL_SAM_ENABLED to False.
ModelDependencyMissing: Your `inference` configuration does not support SAM3 model. Install SAM3 dependencies and set CORE_MODEL_SAM3_ENABLED to True.
ModelDependencyMissing: Your `inference` configuration does not support Gaze Detection model. Use pip install 'inference[gaze]' to install missing requirements.To suppress this warning, set CORE_MODEL_GAZE_ENABLED to False.
ModelDependencyMissing: Your `inference` configuration does not support YoloWorld model. Use pip install 'inference[yolo-world]' to install missing requirements.To suppress this warning, set CORE_MODEL_YOLO_WORLD_ENABLED to False.


Model loaded successfully!


## Step 4 — Verify the saved files

In [4]:
import glob

saved_files = glob.glob(os.path.join(MODEL_SAVE_PATH, "**"), recursive=True)
saved_files = [f for f in saved_files if os.path.isfile(f)]

if saved_files:
    print(f"Found {len(saved_files)} file(s) saved in {MODEL_SAVE_PATH}:\n")
    for f in saved_files:
        size_kb = os.path.getsize(f) / 1024
        print(f"  {os.path.relpath(f, MODEL_SAVE_PATH):<50}  {size_kb:>8.1f} KB")
else:
    print("WARNING: No files found. The cache dir may not have been respected.")
    print("Falling back to Step 5 — manual copy approach.")

Falling back to Step 5 — manual copy approach.


## Step 5 — Fallback: manually find & copy the cached model

If Step 4 shows no files, the `inference` library saved the model somewhere else.
This cell finds it and copies it to your target directory.

In [5]:
import shutil

# Common cache locations used by the inference library
CANDIDATE_CACHE_DIRS = [
    os.path.expanduser("~/.cache/roboflow"),
    "/tmp/cache",
    "/tmp/roboflow",
    os.path.join(os.path.expanduser("~"), ".inference"),
]

found_dir = None
for candidate in CANDIDATE_CACHE_DIRS:
    if os.path.exists(candidate):
        contents = os.listdir(candidate)
        if contents:
            print(f"Found cache at: {candidate}")
            print(f"  Contents: {contents}")
            found_dir = candidate
            break

if found_dir:
    dest = os.path.join(MODEL_SAVE_PATH, "cache_copy")
    shutil.copytree(found_dir, dest, dirs_exist_ok=True)
    print(f"\nCopied cache → {dest}")
else:
    print("Could not auto-locate the cache. Run the cell below to search your filesystem.")

Found cache at: /tmp/cache
  Contents: ['weapondetection-xx3lz-tgh2t', 'usage.db', '_file_locks']

Copied cache → /media/yousifcreates/339b14bb-e7ae-4348-b1dd-b0b8a896b600/Portfolio/Deep Learning/ThreatVision/ThreatVision/model/weapondetection_v2/cache_copy


## Step 6 — Deep search (last resort)
Only run this if Steps 4 and 5 both failed. It searches your home directory for `.onnx` / `.pt` files that match the model.

In [ ]:
search_root = os.path.expanduser("~")
print(f"Searching for model weight files under {search_root} ...\n")

extensions = (".onnx", ".pt", ".pth", ".weights")

for root, dirs, files in os.walk(search_root):
    # Skip large irrelevant dirs
    dirs[:] = [d for d in dirs if d not in (".git", "node_modules", "__pycache__", ".venv")]
    for fname in files:
        if fname.endswith(extensions):
            full = os.path.join(root, fname)
            size_mb = os.path.getsize(full) / (1024 * 1024)
            print(f"  {full}  ({size_mb:.1f} MB)")

## Step 7 — Quick smoke test
Verify the saved model can run inference on a blank frame (no image needed).

In [6]:
import numpy as np
import supervision as sv

# Create a blank 640x640 test frame
test_frame = np.zeros((640, 640, 3), dtype=np.uint8)

results    = model.infer(test_frame, confidence=0.4)[0]
detections = sv.Detections.from_inference(results)

print(f"Smoke test passed!")
print(f"Detections on blank frame: {len(detections)}  (expected: 0)")
print(f"\nModel is ready. Local save path to use in backend_logic.py:")
print(f"  MODEL_SAVE_PATH = \"{MODEL_SAVE_PATH}\"")

Smoke test passed!
Detections on blank frame: 0  (expected: 0)

Model is ready. Local save path to use in backend_logic.py:
  MODEL_SAVE_PATH = "/media/yousifcreates/339b14bb-e7ae-4348-b1dd-b0b8a896b600/Portfolio/Deep Learning/ThreatVision/ThreatVision/model/weapondetection_v2"


---
## ✅ Done!
Once the smoke test passes, your model is saved locally. Share the `MODEL_SAVE_PATH` value printed above — we'll use it in the updated `backend_logic.py` to load from disk on every run.